In [1]:
!pip install /kaggle/input/pip-install-lifelines/autograd-1.7.0-py3-none-any.whl
!pip install /kaggle/input/pip-install-lifelines/autograd-gamma-0.5.0.tar.gz
!pip install /kaggle/input/pip-install-lifelines/interface_meta-1.3.0-py3-none-any.whl
!pip install /kaggle/input/pip-install-lifelines/formulaic-1.0.2-py3-none-any.whl
!pip install /kaggle/input/pip-install-lifelines/lifelines-0.30.0-py3-none-any.whl

Processing /kaggle/input/pip-install-lifelines/autograd-1.7.0-py3-none-any.whl
autograd is already installed with the same version as the provided wheel. Use --force-reinstall to force an installation of the wheel.
Processing /kaggle/input/pip-install-lifelines/autograd-gamma-0.5.0.tar.gz
  Preparing metadata (setup.py) ... done
  Created wheel for autograd-gamma: filename=autograd_gamma-0.5.0-py3-none-any.whl size=4031 sha256=bb28f19be5d1fe0e834bdaff92174fcd7ae9b878c234302a1d47d8963be179c9
  Stored in directory: /root/.cache/pip/wheels/6b/b5/e0/4c79e15c0b5f2c15ecf613c720bb20daab20a666eb67135155
Successfully built autograd-gamma
Processing /kaggle/input/pip-install-lifelines/interface_meta-1.3.0-py3-none-any.whl
Processing /kaggle/input/pip-install-lifelines/formulaic-1.0.2-py3-none-any.whl
Processing /kaggle/input/pip-install-lifelines/lifelines-0.30.0-py3-none-any.whl


In [2]:
import warnings
warnings.simplefilter(action='ignore')

import os
import requests
import joblib

import numpy
import pandas

import matplotlib.pyplot as plt
import scipy.stats as stats
import tensorflow_decision_forests as tfdf

In [3]:
from catboost import CatBoostRegressor ,Pool
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor

from sklearn.linear_model import ElasticNetCV, LassoCV, RidgeCV
from sklearn.svm import SVR

from sklearn.ensemble import GradientBoostingRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.ensemble import StackingRegressor
from sklearn.ensemble import AdaBoostRegressor

from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import RobustScaler
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import MinMaxScaler

from sklearn.metrics import mean_squared_error
from sklearn.metrics import mean_absolute_error
from sklearn.metrics import accuracy_score

from sklearn.pipeline import make_pipeline
from sklearn.model_selection import KFold, cross_val_score

from sklearn.feature_selection import SelectKBest
from sklearn.feature_selection import f_classif
from sklearn.feature_selection import SelectFromModel

from sklearn.impute import SimpleImputer
import sklearn.linear_model as linear_model
from sklearn.model_selection import train_test_split
import seaborn as sns
from sklearn.manifold import TSNE
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA

In [4]:
train = pandas.read_csv('/kaggle/input/equity-post-HCT-survival-predictions/train.csv').drop(columns=['ID'])
test = pandas.read_csv('/kaggle/input/equity-post-HCT-survival-predictions/test.csv').drop(columns=['ID'])

In [5]:
from lifelines import KaplanMeierFitter

def calculate_survival_probabilities(df, time_col, event_col):
    kmf = KaplanMeierFitter()
    kmf.fit(df[time_col], df[event_col])
    return kmf.survival_function_at_times(df[time_col]).values

def preprocess_survival_data(df, time_col='efs_time', event_col='efs'):
    df['target'] = calculate_survival_probabilities(df, time_col, event_col)
    df.loc[df[event_col] == 0, 'target'] -= 0.2  # Adjust for censored data
    return df

train = preprocess_survival_data(train)

In [6]:
encoder = LabelEncoder()
normalize = SimpleImputer(strategy='mean')

In [7]:
for i in zip(train.columns,train.dtypes):
    if i[1]=='O':
        train[i[0]] = train[i[0]].fillna('Unknown')
        #train[i[0]]=encoder.fit_transform(train[i[0]].to_numpy().reshape(-1,1))
    else:
        train[i[0]] = train[i[0]].fillna(0)
        #train[i[0]]=normalize.fit_transform(numpy.array(train[i[0]]).reshape(-1,1))
for i in zip(test.columns,test.dtypes):
    if i[1]=='O':
        test[i[0]] = test[i[0]].fillna('Unknown')
        #test[i[0]]=encoder.fit_transform(test[i[0]].to_numpy().reshape(-1,1))
    else:
        test[i[0]] = test[i[0]].fillna(0)
        #test[i[0]]=normalize.fit_transform(numpy.array(test[i[0]]).reshape(-1,1))

In [8]:
train_x = train.drop(columns=['efs', 'efs_time', 'target'])
train_y = train['target']
    
#train.hist(figsize=(16, 20), bins=50, xlabelsize=8, ylabelsize=8)

#clf = GradientBoostingRegressor(n_estimators=100).fit(train_x, train_y)
#model = SelectFromModel(clf, prefit=True)

#weight = model.transform(train_x).reshape(-1)
train.head()

,dri_score,psych_disturb,cyto_score,diabetes,hla_match_c_high,hla_high_res_8,tbi_status,arrhythmia,hla_low_res_6,graft_type,...,donor_related,melphalan_dose,hla_low_res_8,cardiac,hla_match_drb1_high,pulm_moderate,hla_low_res_10,efs,efs_time,target
0,N/A - non-malignant indication,No,Unknown,No,0.0,0.0,No TBI,No,6.0,Bone marrow,...,Unrelated,"N/A, Mel not given",8.0,No,2.0,No,10.0,0.0,42.356,0.258687
1,Intermediate,No,Intermediate,No,2.0,8.0,"TBI +- Other, >cGy",No,6.0,Peripheral blood,...,Related,"N/A, Mel not given",8.0,No,2.0,Yes,10.0,1.0,4.672,0.847759
2,N/A - non-malignant indication,No,Unknown,No,2.0,8.0,No TBI,No,6.0,Bone marrow,...,Related,"N/A, Mel not given",8.0,No,2.0,No,10.0,0.0,19.793,0.262424
3,High,No,Intermediate,No,2.0,8.0,No TBI,No,6.0,Bone marrow,...,Unrelated,"N/A, Mel not given",8.0,No,2.0,No,10.0,0.0,102.349,0.256661
4,High,No,Unknown,No,2.0,8.0,No TBI,No,6.0,Peripheral blood,...,Related,MEL,8.0,No,2.0,No,10.0,0.0,16.223,0.264674


In [9]:
Fold=5
kf = KFold(n_splits=Fold)

cat_pre_train = numpy.zeros(len(train))
cat_pre_test = numpy.zeros(len(test))

catboost = CatBoostRegressor(grow_policy='Depthwise',
                             iterations=1000,
                             eval_metric='RMSE',
                             random_state=0,
                             verbose=100)

for i, (train_index, test_index) in enumerate(kf.split(train)):

    print(f"FOLD: {i}")

    x_fold=train_x[train_index[0]: train_index[len(train_index)-1]]
    y_fold=train_y[train_index[0]: train_index[len(train_index)-1]]

    x_test_fold=train_x[test_index[0]: test_index[len(test_index)-1]]
    y_test_fold=train_y[test_index[0]: test_index[len(test_index)-1]]

    cat_features = list(train_x.select_dtypes(include=['object', 'category']).columns)
    
    train_pool = Pool(x_fold, y_fold, cat_features=cat_features)
    val_pool = Pool(x_test_fold, y_test_fold, cat_features=cat_features)
    
    #weight_ = weight[train_index[0]: train_index[len(train_index)-1]]
    
    catboost.fit(train_pool, eval_set=(val_pool), verbose=50, early_stopping_rounds=100)

    cat_pre_train+=catboost.predict(train_x)
    
cat_pre_test=catboost.predict(test)
print(catboost.score(x_test_fold, y_test_fold))

FOLD: 0
Learning rate set to 0.08333
0:	learn: 0.2583050	test: 0.2571605	best: 0.2571605 (0)	total: 111ms	remaining: 1m 50s
50:	learn: 0.2344231	test: 0.2359324	best: 0.2359324 (50)	total: 1.6s	remaining: 29.7s
100:	learn: 0.2291415	test: 0.2333193	best: 0.2333193 (100)	total: 2.98s	remaining: 26.5s
150:	learn: 0.2245564	test: 0.2322804	best: 0.2322804 (150)	total: 4.27s	remaining: 24s
200:	learn: 0.2213122	test: 0.2318080	best: 0.2317904 (197)	total: 5.48s	remaining: 21.8s
250:	learn: 0.2183554	test: 0.2312601	best: 0.2312601 (250)	total: 6.68s	remaining: 19.9s
300:	learn: 0.2158608	test: 0.2312837	best: 0.2312589 (252)	total: 7.85s	remaining: 18.2s
350:	learn: 0.2136336	test: 0.2310632	best: 0.2310596 (340)	total: 8.99s	remaining: 16.6s
400:	learn: 0.2115857	test: 0.2309386	best: 0.2309316 (396)	total: 10.1s	remaining: 15s
450:	learn: 0.2098777	test: 0.2309437	best: 0.2308941 (440)	total: 11.1s	remaining: 13.6s
500:	learn: 0.2080597	test: 0.2309745	best: 0.2308941 (440)	total: 12.2s	

In [10]:
id = pandas.read_csv('/kaggle/input/equity-post-HCT-survival-predictions/test.csv')['ID']
test_predictions = (cat_pre_test)
#((RandomF_pre_test/Fold)+(LGBM_pre_test/Fold)+(xgb_pre_test/Fold)+(cat_pre_test/Fold))/4
#model.predict(new_test_data)
#((RandomF_pre_test/10)+(LGBM_pre_test/5)+(xgb_pre_test/5)+(cat_pre_test/5))/4
test_predictions

array([0.31972981, 0.60372347, 0.25311782])

In [11]:
submission = pandas.DataFrame({
    'ID': id.values,
    'prediction': test_predictions,
})
submission

,ID,prediction
0,28800,0.319730
1,28801,0.603723
2,28802,0.253118


In [12]:
submission.to_csv('submission.csv', index = False)